# Reliability Gate — Combining Uncertainty and Sensor Consistency

Step 5: for a given 30-cycle window, ACCEPT the RUL prediction only if the BiLSTM's MC-Dropout uncertainty is below its calibrated threshold **and** no monitored sensor shows a persistent consistency violation. Otherwise ABSTAIN, with the specific reason(s) reported -- exactly matching the live demo spec.

Uses uncapped RUL, on purpose: this gate's whole job is judging whether a reading can be trusted, and capped RUL would hide the early-life scenario where that judgment matters most.

Requires `consistency_models.pkl` from `sensor_consistency.ipynb` to already exist.

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

C:\Users\Shivam\AppData\Roaming\Python\Python310\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.2) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


## Load data, reconstruct uncapped RUL, same split as everywhere else

In [2]:
df = pd.read_csv("Dataset/processed_train.csv")

# processed_train.csv has RUL capped at 125. Reconstruct the true, uncapped
# RUL directly from Cycle -- capping only clips the RUL *value*, it never
# drops rows, so Max_Cycle per engine is unaffected.
max_cycle = df.groupby("Engine_ID")["Cycle"].transform("max")
df["RUL"] = max_cycle - df["Cycle"]

np.random.seed(42)
engine_ids = df["Engine_ID"].unique()
np.random.shuffle(engine_ids)

train_ids = engine_ids[:70]
val_ids = engine_ids[70:85]
test_ids = engine_ids[85:]

train_df = df[df["Engine_ID"].isin(train_ids)].copy()
val_df = df[df["Engine_ID"].isin(val_ids)].copy()
test_df = df[df["Engine_ID"].isin(test_ids)].copy()

print("Training engines:", train_df["Engine_ID"].nunique())
print("Validation engines:", val_df["Engine_ID"].nunique())
print("Testing engines:", test_df["Engine_ID"].nunique())

Training engines: 70
Validation engines: 15
Testing engines: 15


In [3]:
FEATURES = [
    "Setting_1", "Setting_2",
    "Sensor_2", "Sensor_3", "Sensor_4", "Sensor_6", "Sensor_7", "Sensor_8", "Sensor_9",
    "Sensor_11", "Sensor_12", "Sensor_13", "Sensor_14", "Sensor_15", "Sensor_17",
    "Sensor_20", "Sensor_21"
]
TARGET = "RUL"
WINDOW = 30

def create_sequences(data, features, target, window=30):
    X, y = [], []
    for engine_id, engine_data in data.groupby("Engine_ID"):
        engine_data = engine_data.sort_values("Cycle")
        feature_values = engine_data[features].values.astype("float32")
        target_values = engine_data[target].values.astype("float32")
        for i in range(window, len(engine_data)):
            X.append(feature_values[i-window:i])
            y.append(target_values[i])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_df, FEATURES, TARGET, WINDOW)
X_val, y_val = create_sequences(val_df, FEATURES, TARGET, WINDOW)

print(X_train.shape, X_val.shape)

(12216, 30, 17) (2720, 30, 17)


## Train the BiLSTM (same architecture as every other notebook in this project)

In [4]:
tf.random.set_seed(42)

bilstm = Sequential([
    Input(shape=(WINDOW, len(FEATURES))),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.2),
    Bidirectional(LSTM(32)),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1)
])

bilstm.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss="mse", metrics=["mae"])

early_stopping = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6)

bilstm.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100, batch_size=64, shuffle=False,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

Epoch 1/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 14s 35ms/step - loss: 7619.9517 - mae: 68.7876 - val_loss: 4184.3555 - val_mae: 51.8275 - learning_rate: 0.0010
Epoch 2/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - loss: 3892.2417 - mae: 50.2320 - val_loss: 3648.1890 - val_mae: 50.0616 - learning_rate: 0.0010
Epoch 3/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - loss: 3821.4287 - mae: 50.2689 - val_loss: 3651.7180 - val_mae: 50.1602 - learning_rate: 0.0010
Epoch 4/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - loss: 3828.2681 - mae: 50.2974 - val_loss: 3651.7166 - val_mae: 50.1606 - learning_rate: 0.0010
Epoch 5/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - loss: 3832.4387 - mae: 50.3745 - val_loss: 3651.4810 - val_mae: 50.1566 - learning_rate: 0.0010
Epoch 6/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - loss: 3844.2920 - mae: 50.4755 - val_loss: 3651.5044 - val_mae: 50.1575 - learning_rate: 0.0010
Epoch 7/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 3821.7546 - mae: 50.2298 

## MC-Dropout uncertainty, calibrated on `val_df`

In [5]:
def mc_dropout_predict(model, X, n_passes=100):
    predictions = np.array([model(X, training=True).numpy().ravel() for _ in range(n_passes)])
    return predictions.mean(axis=0), predictions.std(axis=0)

mc_val_mean, mc_val_std = mc_dropout_predict(bilstm, X_val, n_passes=100)

ACCEPT_RATE = 0.80
mc_threshold = np.quantile(mc_val_std, ACCEPT_RATE)
print("MC uncertainty threshold:", mc_threshold)

MC uncertainty threshold: 10.536944


## Load the consistency models from `sensor_consistency.ipynb`

In [6]:
artifact = joblib.load("consistency_models.pkl")
models = artifact["models"]
sigmas = artifact["sigmas"]
thresholds = artifact["thresholds"]
MONITORED_SENSORS = artifact["monitored_sensors"]
CONSISTENCY_FEATURES = artifact["features"]
PERSISTENCE_WINDOW = artifact["persistence_window"]
PERSISTENCE_MIN_COUNT = artifact["persistence_min_count"]

print("Monitored sensors:", MONITORED_SENSORS)

Monitored sensors: ['Sensor_9', 'Sensor_14', 'Sensor_11', 'Sensor_12', 'Sensor_8', 'Sensor_13', 'Sensor_4', 'Sensor_7', 'Sensor_15', 'Sensor_20', 'Sensor_21', 'Sensor_17', 'Sensor_2', 'Sensor_3']


## Helper functions

`extract_window` pulls exactly one 30-cycle window ending at a given cycle. `consistency_check` runs the persistence rule over the most recent `PERSISTENCE_WINDOW` cycles *within* that window -- since `PERSISTENCE_WINDOW` (7) is smaller than `WINDOW` (30), this needs no history beyond the window itself, matching a real "load the last 30 cycles" deployment.

In [7]:
def extract_window(engine_data, end_cycle, window=30):
    engine_data = engine_data.sort_values("Cycle")
    windowed = engine_data[(engine_data["Cycle"] > end_cycle - window) & (engine_data["Cycle"] <= end_cycle)]
    return windowed.reset_index(drop=True)


def check_persistence(flags, window=7, min_count=5):
    flags = np.asarray(flags).astype(int)
    return pd.Series(flags).rolling(window, min_periods=1).sum().values >= min_count


def consistency_check(window_df, models, sigmas, thresholds, monitored_sensors, features,
                       persistence_window=7, persistence_min_count=5):
    """Runs the persistence rule over the last `persistence_window` cycles of
    this one window. Returns (consistent, worst_sensor) -- worst_sensor is
    the monitored sensor with the largest normalized residual among any that
    tripped persistence, or None if everything looks normal."""
    recent = window_df.tail(persistence_window)

    consistent = True
    worst_sensor = None
    worst_residual = -np.inf

    for sensor in monitored_sensors:
        predictors = [f for f in features if f != sensor]
        pred = models[sensor].predict(recent[predictors])
        residual = np.abs(recent[sensor].values - pred) / sigmas[sensor]
        flags = residual > thresholds[sensor]
        persistent = check_persistence(flags, persistence_window, persistence_min_count)

        if persistent.any():
            consistent = False
            if residual.max() > worst_residual:
                worst_residual = residual.max()
                worst_sensor = sensor

    return consistent, worst_sensor

## Fault injection (same definitions as `fault_injection_eval.ipynb`)

In [8]:
def inject_gradual_drift(engine_data, sensor, start_cycle, severity_per_cycle):
    engine_data = engine_data.copy()
    mask = engine_data["Cycle"] >= start_cycle
    steps = (engine_data.loc[mask, "Cycle"] - start_cycle + 1).values
    engine_data.loc[mask, sensor] = engine_data.loc[mask, sensor].values + severity_per_cycle * steps
    return engine_data


def inject_stuck_at(engine_data, sensor, start_cycle):
    engine_data = engine_data.copy()
    mask = engine_data["Cycle"] >= start_cycle
    frozen_value = engine_data.loc[engine_data["Cycle"] == start_cycle - 1, sensor]
    if len(frozen_value) == 0:
        frozen_value = engine_data.loc[engine_data["Cycle"] == start_cycle, sensor]
    frozen_value = frozen_value.values[0]
    engine_data.loc[mask, sensor] = frozen_value
    return engine_data

SEVERE_SEVERITY = 0.01

## Evaluate the gate on clean test engines

One window per engine -- the last 30 cycles, matching the live demo's "load last 30 sensor cycles" exactly. This is what a healthy, trustworthy reading should look like: mostly ACCEPT.

In [9]:
clean_windows_X, clean_windows_df, clean_engine_ids = [], [], []

for engine_id, engine_data in test_df.groupby("Engine_ID"):
    engine_data = engine_data.sort_values("Cycle")
    if len(engine_data) < WINDOW:
        continue
    end_cycle = engine_data["Cycle"].max()
    window_df = extract_window(engine_data, end_cycle, WINDOW)
    clean_windows_X.append(window_df[FEATURES].values.astype("float32"))
    clean_windows_df.append(window_df)
    clean_engine_ids.append(engine_id)

clean_X = np.array(clean_windows_X)
clean_mc_mean, clean_mc_std = mc_dropout_predict(bilstm, clean_X, n_passes=100)
print("Clean windows evaluated:", len(clean_engine_ids))

Clean windows evaluated: 15


In [10]:
clean_decisions = []
for i in range(len(clean_engine_ids)):
    consistency_ok, worst_sensor = consistency_check(
        clean_windows_df[i], models, sigmas, thresholds, MONITORED_SENSORS, CONSISTENCY_FEATURES,
        PERSISTENCE_WINDOW, PERSISTENCE_MIN_COUNT
    )
    uncertainty_ok = clean_mc_std[i] <= mc_threshold
    clean_decisions.append("ACCEPT" if (uncertainty_ok and consistency_ok) else "ABSTAIN")

clean_decisions = np.array(clean_decisions)
print(f"Accept rate on clean engines: {(clean_decisions == 'ACCEPT').mean():.1%} "
      f"({(clean_decisions == 'ACCEPT').sum()}/{len(clean_decisions)})")

Accept rate on clean engines: 100.0% (15/15)


## Evaluate the gate under severe faults

Same test engines, but the fault starts exactly at the beginning of the final window, so the whole 30-cycle window the gate sees is fault-affected -- the scenario the gate exists to catch. Severe severity only, to keep this readable; swap in mild or add more onsets the same way `fault_injection_eval.ipynb` does if you want a fuller sweep.

In [11]:
fault_windows_X, fault_windows_df, fault_meta = [], [], []

for engine_id, engine_data in test_df.groupby("Engine_ID"):
    engine_data = engine_data.sort_values("Cycle").reset_index(drop=True)
    if len(engine_data) < WINDOW:
        continue
    end_cycle = engine_data["Cycle"].max()
    onset = end_cycle - WINDOW + 1

    for sensor in MONITORED_SENSORS:
        for fault_type in ["drift_up", "drift_down", "stuck_at"]:
            if fault_type == "drift_up":
                faulted = inject_gradual_drift(engine_data, sensor, onset, SEVERE_SEVERITY)
            elif fault_type == "drift_down":
                faulted = inject_gradual_drift(engine_data, sensor, onset, -SEVERE_SEVERITY)
            else:
                faulted = inject_stuck_at(engine_data, sensor, onset)

            window_df = extract_window(faulted, end_cycle, WINDOW)
            fault_windows_X.append(window_df[FEATURES].values.astype("float32"))
            fault_windows_df.append(window_df)
            fault_meta.append({"Engine_ID": engine_id, "Sensor": sensor, "Fault_type": fault_type})

fault_X = np.array(fault_windows_X)
fault_mc_mean, fault_mc_std = mc_dropout_predict(bilstm, fault_X, n_passes=100)
print("Fault windows evaluated:", len(fault_meta))

Fault windows evaluated: 630


In [12]:
fault_decisions = []
for i in range(len(fault_meta)):
    consistency_ok, worst_sensor = consistency_check(
        fault_windows_df[i], models, sigmas, thresholds, MONITORED_SENSORS, CONSISTENCY_FEATURES,
        PERSISTENCE_WINDOW, PERSISTENCE_MIN_COUNT
    )
    uncertainty_ok = fault_mc_std[i] <= mc_threshold
    fault_decisions.append("ACCEPT" if (uncertainty_ok and consistency_ok) else "ABSTAIN")

fault_results = pd.DataFrame(fault_meta)
fault_results["Decision"] = fault_decisions

print(f"Abstain rate under severe, fully-injected faults: {(fault_results['Decision']=='ABSTAIN').mean():.1%}")
print()
print(fault_results.groupby("Fault_type")["Decision"].value_counts(normalize=True).unstack())

Abstain rate under severe, fully-injected faults: 69.0%

Decision     ABSTAIN    ACCEPT
Fault_type                    
drift_down  0.861905  0.138095
drift_up    0.895238  0.104762
stuck_at    0.314286  0.685714


In [13]:
fault_results.to_csv("reliability_gate_results.csv", index=False)
print("Saved reliability_gate_results.csv")

Saved reliability_gate_results.csv
